# CISC3014 音乐推荐系统 - 数据分析

本 Notebook 用于：
1. 加载和探索爬取的音乐数据
2. 训练推荐算法模型
3. 生成推荐结果
4. 可视化分析

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# 添加项目根目录到路径
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.append(project_root)

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 设置绘图样式
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8')

print('库导入完成！')

## 1. 数据加载与探索

In [ ]:
# 加载数据
# TODO: 替换为实际的数据文件路径
data_path = '../data/raw/music_data.csv'

# 示例：创建模拟数据
np.random.seed(42)
n_users = 100
n_items = 50

# 生成模拟的用户-评分数据
data = []
for user_id in range(n_users):
    # 每个用户随机评分10-20个物品
    n_ratings = np.random.randint(10, 20)
    items = np.random.choice(n_items, n_ratings, replace=False)
    ratings = np.random.randint(1, 6, n_ratings)
    for item, rating in zip(items, ratings):
        data.append({'user_id': f'user_{user_id}', 'item_id': f'song_{item}', 'rating': rating})

df = pd.DataFrame(data)
print(f'数据形状: {df.shape}')
df.head(10)

In [ ]:
# 数据统计
print(f'用户数量: {df["user_id"].nunique()}')
print(f'歌曲数量: {df["item_id"].nunique()}')
print(f'评分数量: {len(df)}')
print(f'平均评分: {df["rating"].mean():.2f}')

# 评分分布
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='rating', palette='viridis')
plt.title('评分分布')
plt.xlabel('评分')
plt.ylabel('数量')
plt.show()

## 2. 构建推荐模型

In [ ]:
from src.analysis.recommender import CollaborativeFiltering

# 训练基于用户的协同过滤模型
cf_model = CollaborativeFiltering(method='user')
cf_model.fit(df)

print('模型训练完成！')

## 3. 生成推荐

In [ ]:
# 为指定用户生成推荐
user_id = 'user_0'
recommendations = cf_model.recommend(user_id, top_n=10)

print(f'为用户 {user_id} 推荐的歌曲:')
for i, (song_id, score) in enumerate(recommendations, 1):
    print(f'{i}. {song_id} (预测评分: {score:.2f})')

## 4. 结果可视化

In [ ]:
# 可视化推荐结果
songs = [r[0] for r in recommendations]
scores = [r[1] for r in recommendations]

plt.figure(figsize=(12, 6))
colors = sns.color_palette('viridis', len(songs))
plt.barh(songs, scores, color=colors)
plt.xlabel('预测评分')
plt.title(f'为用户 {user_id} 推荐的歌曲')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. 保存结果

In [ ]:
# 保存推荐结果
result_df = pd.DataFrame(recommendations, columns=['song_id', 'predicted_rating'])
result_df.to_csv('../output/recommendations.csv', index=False)
print('推荐结果已保存到 ../output/recommendations.csv')